task # 1

In [7]:
import math
from collections import Counter

def entropy(labels):
    if len(labels) == 0:
        return 0.0
    counts = Counter(labels)
    total = len(labels)
    result = 0.0
    for count in counts.values():
        p = count / total
        if p > 0:
            result -= p * math.log2(p)
    return result

labels1 = ["Pass", "Pass", "Pass", "Pass"]
labels2 = ["Pass", "Pass", "Fail", "Fail"]
labels3 = ["Pass", "Fail", "Pass", "Fail"]

h1 = entropy(labels1)
h2 = entropy(labels2)
h3 = entropy(labels3)

print(f"Entropy of labels1: {h1:.4f} bits")
print(f"Entropy of labels2: {h2:.4f} bits")
print(f"Entropy of labels3: {h3:.4f} bits")

Entropy of labels1: 0.0000 bits
Entropy of labels2: 1.0000 bits
Entropy of labels3: 1.0000 bits


task # 2

In [8]:
from collections import defaultdict

dataset = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "No",  "Label": "Pass"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "Yes", "Label": "Pass"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "No",  "Label": "Fail"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "Yes", "Label": "Fail"},
    {"Attendance": "High", "Study": "High", "Assignment": "No",  "Label": "Pass"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "No",  "Label": "Fail"}
]

def conditional_entropy(dataset, feature, label_col="Label"):
    total = len(dataset)
    if total == 0:
        return 0.0
    subsets = defaultdict(list)
    for row in dataset:
        subsets[row[feature]].append(row[label_col])
    return sum((len(subset) / total) * entropy(subset) for subset in subsets.values())

def mutual_information(dataset, feature, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    h_y = entropy(labels)
    h_y_given_x = conditional_entropy(dataset, feature, label_col)
    return h_y - h_y_given_x

features = ["Attendance", "Study", "Assignment"]
h_y = entropy([row["Label"] for row in dataset])
print(f"Dataset H(Y): {h_y:.4f} bits\n")

for f in features:
    cond_h = conditional_entropy(dataset, f)
    mi = mutual_information(dataset, f)
    print(f"{f:<12} | Conditional Entropy H(Y|{f}): {cond_h:.4f} | MI: {mi:.4f} bits")

best_feature = max(features, key=lambda f: mutual_information(dataset, f))
print(f"\nBest feature to split on: {best_feature}")

Dataset H(Y): 0.9544 bits

Attendance   | Conditional Entropy H(Y|Attendance): 0.4056 | MI: 0.5488 bits
Study        | Conditional Entropy H(Y|Study): 0.9056 | MI: 0.0488 bits
Assignment   | Conditional Entropy H(Y|Assignment): 0.9056 | MI: 0.0488 bits

Best feature to split on: Attendance


task # 3

In [9]:
class Node:
    def __init__(self, node_type, label=None, feature=None):
        self.type = node_type        # "leaf" or "internal"
        self.label = label          # used by leaf nodes
        self.feature = feature      # used by internal nodes
        self.branches = {}          # branch_value -> child Node

def predict(tree, x):
    current = tree
    while current.type != "leaf":
        feature = current.feature
        value = x[feature]
        current = current.branches[value]
    return current.label

fail_leaf = Node("leaf", label="Fail")
pass_leaf = Node("leaf", label="Pass")

study_node = Node("internal", feature="Study")
study_node.branches["Low"] = fail_leaf
study_node.branches["High"] = pass_leaf

root = Node("internal", feature="Attendance")
root.branches["Low"] = fail_leaf
root.branches["High"] = study_node

test_students = [
    {"Attendance": "High", "Study": "High"},
    {"Attendance": "High", "Study": "Low"},
    {"Attendance": "Low",  "Study": "High"}
]

print("Manual Tree Predictions:")
for i, student in enumerate(test_students, 1):
    pred = predict(root, student)
    print(f"Student {i} {student} -> Prediction: {pred}")

Manual Tree Predictions:
Student 1 {'Attendance': 'High', 'Study': 'High'} -> Prediction: Pass
Student 2 {'Attendance': 'High', 'Study': 'Low'} -> Prediction: Fail
Student 3 {'Attendance': 'Low', 'Study': 'High'} -> Prediction: Fail


task # 4

In [10]:
def majority_label(dataset, label_col="Label"):
    labels = [row[label_col] for row in dataset]
    return Counter(labels).most_common(1)[0][0]

def train(dataset, features, label_col="Label"):
    # Base Case 1: all labels are identical
    labels = [row[label_col] for row in dataset]
    if len(set(labels)) == 1:
        return Node("leaf", label=labels[0])
    
    # Base Case 2: no features remain
    if len(features) == 0:
        return Node("leaf", label=majority_label(dataset, label_col))
    
    # Select feature with maximum Mutual Information
    best_feature = max(
        features,
        key=lambda f: mutual_information(dataset, f, label_col)
    )
    
    # In case MI is 0 for all features (cannot split further)
    if mutual_information(dataset, best_feature, label_col) == 0:
        return Node("leaf", label=majority_label(dataset, label_col))
        
    node = Node("internal", feature=best_feature)
    
    # Partition data on best_feature values
    values = set(row[best_feature] for row in dataset)
    remaining_features = [f for f in features if f != best_feature]
    
    for value in values:
        subset = [row for row in dataset if row[best_feature] == value]
        child = train(subset, remaining_features, label_col)
        node.branches[value] = child
        
    return node

# Train on the provided dataset
features = ["Attendance", "Study", "Assignment"]
trained_tree = train(dataset, features)

# Helper function to display the tree hierarchy
def display_tree(node, depth=0):
    indent = "  " * depth
    if node.type == "leaf":
        print(f"{indent}--> Leaf [Label: {node.label}]")
    else:
        print(f"{indent}[Split on: {node.feature}]")
        for val, child in sorted(node.branches.items()):
            print(f"{indent}  Branch '{val}':")
            display_tree(child, depth + 2)

print("Trained Decision Tree:")
display_tree(trained_tree)

Trained Decision Tree:
[Split on: Attendance]
  Branch 'High':
    --> Leaf [Label: Pass]
  Branch 'Low':
    [Split on: Study]
      Branch 'High':
        [Split on: Assignment]
          Branch 'No':
            --> Leaf [Label: Fail]
          Branch 'Yes':
            --> Leaf [Label: Pass]
      Branch 'Low':
        --> Leaf [Label: Fail]


task # 5

In [11]:
new_students = [
    {"Attendance": "High", "Study": "High", "Assignment": "Yes"},
    {"Attendance": "High", "Study": "Low",  "Assignment": "No"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "Yes"},
    {"Attendance": "Low",  "Study": "High", "Assignment": "No"},
    {"Attendance": "Low",  "Study": "Low",  "Assignment": "Yes"}
]

print("Predictions on 5 New Student Records:")
print("-" * 60)
for i, student in enumerate(new_students, 1):
    result = predict(trained_tree, student)
    print(f"Student {i}: Attendance={student['Attendance']}, Study={student['Study']}, Assignment={student['Assignment']} => {result}")
print("-" * 60)

Predictions on 5 New Student Records:
------------------------------------------------------------
Student 1: Attendance=High, Study=High, Assignment=Yes => Pass
Student 2: Attendance=High, Study=Low, Assignment=No => Pass
Student 3: Attendance=Low, Study=High, Assignment=Yes => Pass
Student 4: Attendance=Low, Study=High, Assignment=No => Fail
Student 5: Attendance=Low, Study=Low, Assignment=Yes => Fail
------------------------------------------------------------
